In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ---------------------------------------------------------
# Dataset for loading PSD numpy files w/ normalization
# ---------------------------------------------------------
class PSDDataset(Dataset):
    def __init__(self, root_dir, index_file):
        self.items = []
        self.root = root_dir

        with open(index_file, "r") as f:
            for line in f:
                rel_path, lbl = line.strip().split()
                full = os.path.join(self.root, rel_path)
                self.items.append((full, int(lbl)))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        npy_path, label = self.items[idx]
        arr = np.load(npy_path)

        # min-max scale per sample
        mn, mx = arr.min(), arr.max()
        arr = (arr - mn) / (mx - mn + 1e-6)

        arr = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
        return arr, label

# ---------------------------------------------------------
# CNN architecture that adapts to input size at runtime
# ---------------------------------------------------------
class TinyCNN(nn.Module):
    def __init__(self, num_classes=9, sample_shape=(1, 64, 64)):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Determine FC input size using a fake forward pass
        with torch.no_grad():
            test_x = torch.zeros(1, *sample_shape)
            feat = self.block3(self.block2(self.block1(test_x)))
            flat_sz = feat.numel()

        self.head = nn.Sequential(
            nn.Linear(flat_sz, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = torch.flatten(x, 1)
        return self.head(x)

# ---------------------------------------------------------
# Training loop
# ---------------------------------------------------------
def run_training():
    base = "/anvil/projects/x-cis220051/corporate/aerospace-rf/fiot_highway2-main"
    index_file = os.path.join(base, "train.txt")

    batch = 32
    epochs = 10
    lr = 1e-3
    n_classes = 9
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data = PSDDataset(base, index_file)
    if len(data) == 0:
        print("No samples found. Check file paths.")
        return

    # determine input shape dynamically
    first_x, _ = data[0]
    print("Detected input shape:", first_x.shape)

    # split into training/validation
    tr_len = int(0.8 * len(data))
    val_len = len(data) - tr_len
    train_set, val_set = random_split(data, [tr_len, val_len])

    train_loader = DataLoader(train_set, batch_size=batch, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch)

    model = TinyCNN(num_classes=n_classes, sample_shape=first_x.shape).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimzr = optim.Adam(model.parameters(), lr=lr)

    for ep in range(epochs):
        model.train()
        total_loss, hits, samples = 0, 0, 0

        for x, y in tqdm(train_loader, desc=f"Epoch {ep+1}/{epochs} (Training)"):
            x, y = x.to(device), y.to(device)

            optimzr.zero_grad()
            preds = model(x)
            loss = loss_fn(preds, y)
            loss.backward()
            optimzr.step()

            total_loss += loss.item() * x.size(0)
            hits += (preds.argmax(1) == y).sum().item()
            samples += y.size(0)

        train_loss = total_loss / samples
        train_acc = hits / samples

        # ----- Validation -----
        model.eval()
        val_loss, val_hits, val_samples = 0, 0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                ls = loss_fn(out, y)

                val_loss += ls.item() * x.size(0)
                val_hits += (out.argmax(1) == y).sum().item()
                val_samples += y.size(0)

        v_loss = val_loss / val_samples
        v_acc = val_hits / val_samples

        print(
            f"\n┌──────────── Epoch {ep+1}/{epochs} ────────────┐\n"
            f"│   Train → Loss: {train_loss:.4f} | Acc: {train_acc:.4f}   │\n"
            f"│   Val   → Loss: {v_loss:.4f} | Acc: {v_acc:.4f}   │\n"
            f"└──────────────────────────────────────────────┘\n"
        )

    print("Finished training.")


if __name__ == "__main__":
    run_training()

In [ ]:
# visualizing it
import matplotlib.pyplot as plt

# After training loop — ADD THIS BLOCK
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 5))

# ----- Accuracy subplot -----
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_acc_list, label="Train Accuracy", marker="o")
plt.plot(epochs_range, val_acc_list, label="Val Accuracy", marker="o")
plt.title("Accuracy per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, linestyle="--", alpha=0.3)
plt.legend()

# ----- Loss subplot -----
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_loss_list, label="Train Loss", marker="o")
plt.plot(epochs_range, val_loss_list, label="Val Loss", marker="o")
plt.title("Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()